# Pipeline completo — IDS multiclasse SOME/IP (`content_ext`)

Ponta a ponta: **download dos PCAPs → extração de features → split 70/30 → treino → teste →
métricas + matriz de confusão + curvas**. Roda no Colab ou local.

Interruptor **`FROM_SCRATCH`**:
- `False` (padrão): usa as features já versionadas (`data/ours_ext/X.npz`, via Git LFS) — **rápido**.
- `True`: baixa os PCAPs (~1,9 GB do figshare) e **re-extrai do zero** (~6 min) — pipeline completo real.

> Para um experimento só das features (sem download/extração), use `05-ids-multiclasse-content-ext.ipynb`.

In [ ]:
# 0) Setup — no Colab, clona o repo e baixa as features (LFS). Local: assume estar na raiz do repo.
import os
REPO = 'someip-ids-multiclass-contentext'
if not os.path.exists('src/extract_ext.py'):
    if not os.path.isdir(REPO):
        os.system(f'git clone https://github.com/GuilhermeFrick/{REPO}.git')
    os.chdir(REPO)
    os.system('git lfs install && git lfs pull')
print('cwd:', os.getcwd())

FROM_SCRATCH = False   # True = baixa PCAPs + re-extrai do zero

In [ ]:
!pip -q install scapy xgboost scikit-learn matplotlib numpy

In [ ]:
# 1) Download dos PCAPs (só se for do zero, ou se as features não existirem)
import os, glob
need_features = FROM_SCRATCH or not os.path.exists('data/ours_ext/X.npz')
if need_features:
    if len(glob.glob('data/pcap/*.pcap')) < 7:
        !python scripts/download_pcaps.py
    else:
        print('PCAPs já presentes em data/pcap/')
else:
    print('Features já presentes (data/ours_ext/X.npz) — pulando download. Use FROM_SCRATCH=True para refazer.')

In [ ]:
# 2) Extração de features (PCAP -> 21 features content_ext). ~6 min na primeira vez.
if FROM_SCRATCH or not os.path.exists('data/ours_ext/X.npz'):
    !python src/extract_ext.py --pcap-dir data/pcap --out data/ours_ext
else:
    print('Features já presentes — pulando extração.')

In [ ]:
# 3) Carrega features + seleciona content_ext (12 do Kim + 4 comportamentais, sem header)
import numpy as np, json
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc,
                             precision_recall_curve, average_precision_score)
from sklearn.preprocessing import label_binarize
from xgboost import XGBClassifier

CLASSES = ['normal','dos','fuzzy','mitm_single','mitm_multi']; N=len(CLASSES); SEED=0
CONTENT_EXT = list(range(12)) + [12, 13, 14, 16]
names = json.load(open('data/ours_ext/feature_names.json'))
X = np.load('data/ours_ext/X.npz')['a'][:, CONTENT_EXT]
y = np.load('data/ours_ext/y_multi.npz')['a']
print('features:', [names[i] for i in CONTENT_EXT])
print('X:', X.shape, '| classes:', dict(zip(*[v.tolist() for v in np.unique(y, return_counts=True)])))

In [ ]:
# 4) Split 70/30 estratificado + treino XGBoost multiclasse
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
print(f'treino={X_tr.shape[0]:,}  teste={X_te.shape[0]:,}')
clf = XGBClassifier(objective='multi:softprob', num_class=N, n_estimators=300, max_depth=8,
                    learning_rate=0.3, tree_method='hist', max_bin=256, n_jobs=-1, eval_metric='mlogloss')
clf.fit(X_tr, y_tr)
proba = clf.predict_proba(X_te); y_pred = proba.argmax(axis=1)
print('treino concluído')

In [ ]:
# 5) Métricas por classe
print(classification_report(y_te, y_pred, target_names=CLASSES, digits=4))

In [ ]:
# 6) Matriz de confusão
cm = confusion_matrix(y_te, y_pred); cmn = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(7,6)); im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(N)); ax.set_yticks(range(N))
ax.set_xticklabels(CLASSES, rotation=45, ha='right'); ax.set_yticklabels(CLASSES)
ax.set_xlabel('Predito'); ax.set_ylabel('Verdadeiro'); ax.set_title('Matriz de Confusão — content_ext')
for i in range(N):
    for j in range(N):
        ax.text(j, i, f'{cmn[i,j]*100:.1f}%', ha='center', va='center', fontsize=8,
                color='white' if cmn[i,j]>0.5 else 'black')
fig.colorbar(im, fraction=0.046, pad=0.04); plt.tight_layout(); plt.show()

In [ ]:
# 7) Curvas ROC e Precisão-Recall (one-vs-rest)
yb = label_binarize(y_te, classes=range(N))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,6))
for i, name in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve(yb[:,i], proba[:,i]); a = auc(fpr, tpr)
    s = max(len(fpr)//2000,1); ax1.plot(fpr[::s], tpr[::s], lw=1.6, label=f'{name} (AUC={a:.3f})')
    prec, rec, _ = precision_recall_curve(yb[:,i], proba[:,i]); ap = average_precision_score(yb[:,i], proba[:,i])
    s2 = max(len(rec)//2000,1); ax2.plot(rec[::s2], prec[::s2], lw=1.6, label=f'{name} (AP={ap:.3f})')
ax1.plot([0,1],[0,1],'k--',lw=0.7); ax1.set_xlabel('Falso Positivo'); ax1.set_ylabel('Verdadeiro Positivo')
ax1.set_title('ROC (one-vs-rest)'); ax1.legend(fontsize=8, loc='lower right')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precisão'); ax2.set_title('Precisão-Recall (one-vs-rest)')
ax2.legend(fontsize=8, loc='lower left'); plt.tight_layout(); plt.show()

## Leitura

- **macro-F1 ≈ 0,99** — todas as classes F1 ≥ 0,99; ROC-AUC ≈ 1,0.
- Conseguido **sem features de cabeçalho** (só conteúdo + comportamentais), evitando o overfitting
  que features de header causam.
- **Ressalva:** split aleatório 70/30 — número absoluto otimista; o robusto de generalização é o
  zero-day (ver `someip-ensemble-zeroday`).